# SUSY_DRU · arranque en Colab y esqueleto PennyLane

Este notebook deja listo todo lo necesario para hacer una **variante con PennyLane** de la corrida final del proyecto
(7/11/2026) y correrla con más datos, épocas o semillas.

| Paso | Qué hace | Tiempo aprox. |
|---|---|---|
| 1 | Clona el repo `renzovega-creator/susy-dru` e instala PennyLane | 1 min |
| 2 | Baja SUSY (UCI) y verifica que se regenera el split firmado (26 000 filas → 5000/1000/5000) | < 1 min |
| 3 | Reproduce M2, semilla 101, con el motor NumPy del repo y lo compara con los puntajes guardados | 10–20 s |
| 4 | Define el **mismo circuito en PennyLane** y verifica probabilidades y gradientes contra el motor NumPy | < 1 min |
| 5 | Entrena con PennyLane con la misma receta (layerwise + 2 fases + early stopping) | ~1,5 min por semilla con 5000 eventos |
| 6 | Evalúa, compara y guarda | < 1 min |

**Antes de empezar:** entorno de ejecución de CPU; con 1 qubit la GPU no acelera nada.
El repo es público y se clona sin token. Solo para una copia privada: guarda un token de GitHub de solo lectura en los
*Secrets* de Colab con el nombre `GH_TOKEN` (icono de la llave, con *Notebook access*). **No pegues el token en ninguna celda.**

Documentación del protocolo y de los resultados: `corrida_final/README.md` del repo.

## 1 · Repo y dependencias

In [ ]:
import os, sys, subprocess, pathlib

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "github.com/renzovega-creator/susy-dru.git"

if IN_COLAB:
    from google.colab import userdata
    REPO = pathlib.Path("/content/susy-dru")
    if not REPO.exists():
        # El repo es público: el token solo hace falta si se trabaja con una copia privada (Secret GH_TOKEN).
        try:
            token = userdata.get("GH_TOKEN")
        except Exception:
            token = None
        url = f"https://{token}@{REPO_URL}" if token else f"https://{REPO_URL}"
        r = subprocess.run(["git", "clone", "--depth", "1", url, str(REPO)], capture_output=True, text=True)
        if r.returncode:
            raise RuntimeError("git clone falló: " + (r.stderr.replace(token, "***") if token else r.stderr))
        if token:
            # que el token no quede guardado en .git/config
            subprocess.run(["git", "-C", str(REPO), "remote", "set-url", "origin", f"https://{REPO_URL}"], check=True)
        del token
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pennylane"], check=True)
else:
    # Ejecución local: el notebook vive en <repo>/colab/
    REPO = pathlib.Path(os.environ.get("SUSY_DRU_REPO", pathlib.Path.cwd().parent)).resolve()

sys.path[:0] = [str(REPO / "corrida_final"), str(REPO / "replica_dru")]
print("repo:", REPO)
print(subprocess.run(["git", "-C", str(REPO), "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

## 2 · Datos

`N_FILAS` es cuántas filas de `SUSY.csv.gz` se bajan (por streaming, sin bajar los 880 MB completos salvo que lo pidas).
La corrida final usó las **primeras 26 000** de un archivo de 60 000. Para entrenar con más eventos, sube `N_FILAS`
(máximo 5 000 000) y usa `N_TRAIN` / `N_VAL` en el paso 5.

In [ ]:
import hashlib
import numpy as np
import pandas as pd
import susy_data
import corrida_final as cf

N_FILAS = 60_000   # súbelo (p. ej. 200_000) si vas a entrenar con más eventos

def asegurar_csv(n):
    f = REPO / "replica_dru" / "data" / f"susy_head_{n}.csv"
    if not f.exists():
        susy_data.load_susy(n_rows=n)          # descarga y guarda en replica_dru/data/
    return f

f60 = asegurar_csv(60_000)
sha = hashlib.sha256(f60.read_bytes().replace(b"\r\n", b"\n")).hexdigest()
assert sha == "f10ddc24857f874eacc37c3997db3450119032bb9c5a45cbcec29faf7eb721b6", f"CSV distinto al de la corrida: {sha}"

cf.load_data()     # lee las 26 000 filas, regenera el split y lo compara con split_comun.npz (assert)
print({k: len(v) for k, v in cf.IDX.items()}, "· señal en test:", int(cf.Y[cf.IDX["test"]].sum()))

DF_GRANDE = pd.read_csv(asegurar_csv(N_FILAS), header=None, names=cf.COLS)
print(f"{len(DF_GRANDE):,} filas disponibles para el paso 5")

## 3 · Reproducir la corrida final con el motor NumPy

`cf.CONFIGS` tiene las 23 configuraciones pre-registradas (`M1`, `M2`, `M3-lin`, `M3-cua`, `M4-*`, híbridas, `L=8`, ...).
Cada una es un `HP` (dataclass) con todos los hiperparámetros. Aquí se reentrena M2 con la semilla 101 y se compara con
`corrida_final/scores/<hp_key>_101.npz`. En el entorno original la diferencia es 0.0; con otras versiones de NumPy o
scikit-learn pueden aparecer diferencias pequeñas (del orden de 1e-6 o menos), que no cambian el AUC a 4 decimales.

In [ ]:
import time
from sklearn.metrics import roc_auc_score

hp_m2 = cf.CONFIGS["M2"]
print(hp_m2)
t0 = time.perf_counter()
data = cf.prepare(hp_m2)
modelo_np, best_np, hist_np = cf.train(hp_m2, 101, data)
z = np.load(cf.SCORES / f"{cf.hp_key(hp_m2)}_101.npz")
p_te = modelo_np.predict_proba(best_np["p"], data["Xte"])
print(f"{time.perf_counter() - t0:.1f} s · mejor época {best_np['epoch']} · AUC val {best_np['auc']:.4f}")
print("máx |Δ| test vs guardado:", np.max(np.abs(p_te - z["test"])))
print(f"AUC test semilla 101: {roc_auc_score(data['yte'], p_te):.4f} (guardado {roc_auc_score(data['yte'], z['test']):.4f})")

## 4 · El mismo circuito en PennyLane

`ModeloPL` tiene la **misma interfaz** que `dru_core.DRUModel` (`circuit`, `predict_proba`, `loss_and_grad`), pero el
estado lo simula PennyLane y el gradiente sale de `qml.grad` (autograd, backprop). Así el bucle de entrenamiento del
paso 5 es una copia literal del de `corrida_final.train` y cualquier diferencia de resultados se puede atribuir al
simulador. El vector de parámetros también es el mismo: `pesos[l, j]` y `sesgos[l, j]` se leen de las posiciones que
asigna `dru_core.build_circuit("muten", ...)` (mismo mapeo que `replica_dru/test_vs_pennylane.py`).

Circuito por capa (arquitectura "muten", 7 variables): Rot(φ₀, φ₁, φ₂) · Rot(φ₃, φ₄, φ₅) · RY(φ₆), con φⱼ = wⱼ·xⱼ + bⱼ.
Lectura `xyz`: p = sigmoide(a⟨X⟩ + b⟨Y⟩ + c⟨Z⟩ + d). Lectura `z`: p = (1 − ⟨Z⟩)/2.
Pérdidas (las de `dru_core`): `bce`, `mse`, `infidelity` (lineal, ⟨1 − F⟩), `fidelity` (cuadrática, ⟨(1 − F)²⟩),
`hybrid` (α·lineal + (1 − α)·cuadrática), con F = (1 + s⟨Z⟩)/2 y s = +1 para fondo, −1 para señal.

La verificación de abajo compara probabilidades, pérdida y gradiente. Con BCE el gradiente coincide solo hasta
~1e-7 relativo: `dru_core` usa la derivada analítica (p − y)/(p(1 − p)), mientras que aquí autograd deriva
log(p + 1e-9), y ese ε cambia un poco el gradiente cuando p está cerca de 0 o 1. Es la misma sensibilidad de BCE que
anota el README del repo; las demás pérdidas coinciden a precisión de máquina.

In [ ]:
import pennylane as qml
from pennylane import numpy as pnp
from dru_core import DRUModel, build_circuit, Adam, layer_mask

print("PennyLane", qml.__version__)


class ModeloPL:
    def __init__(self, circuit, loss="bce", alpha=0.5, device="default.qubit"):
        assert circuit.n_wires == 1, "este esqueleto es para 1 qubit"
        self.circuit, self.loss, self.alpha = circuit, loss, alpha
        L = circuit.n_layers
        d = 1 + max(g.feat for g in circuit.gates)
        self.n_features = d
        self.iw = np.zeros((L, d), int)
        self.ib = np.zeros((L, d), int)
        for k, g in enumerate(circuit.gates):
            self.iw[circuit.layer_of_gate[k], g.feat] = g.w_idx
            self.ib[circuit.layer_of_gate[k], g.feat] = g.b_idx
        dev = qml.device(device, wires=1)

        @qml.qnode(dev, interface="autograd", diff_method="backprop")
        def bloch(X, w, b):                      # X: (B, d); w, b: (L, d). Broadcasting sobre el lote.
            for l in range(L):
                a = w[l] * X + b[l]
                for t in range(d // 3):
                    qml.Rot(a[:, 3 * t], a[:, 3 * t + 1], a[:, 3 * t + 2], wires=0)
                for j in range(3 * (d // 3), d):
                    qml.RY(a[:, j], wires=0)
            return qml.expval(qml.PauliX(0)), qml.expval(qml.PauliY(0)), qml.expval(qml.PauliZ(0))

        self.bloch = bloch

    def _prob_y_z(self, p, X):
        ex, ey, ez = self.bloch(X, p[self.iw], p[self.ib])
        sl = self.circuit.readout_slice
        if self.circuit.readout == "xyz":
            u = p[sl.start] * ex + p[sl.start + 1] * ey + p[sl.start + 2] * ez + p[sl.start + 3]
            return 0.5 * (1.0 + pnp.tanh(0.5 * u)), ez
        return 0.5 * (1.0 - ez), ez

    def _loss(self, p, X, y):
        prob, ez = self._prob_y_z(p, X)
        s = 1.0 - 2.0 * y
        F = 0.5 * (1.0 + s * ez)
        if self.loss == "bce":
            return pnp.mean(-(y * pnp.log(prob + 1e-9) + (1 - y) * pnp.log(1 - prob + 1e-9)))
        if self.loss == "mse":
            return pnp.mean((prob - y) ** 2)
        if self.loss in ("infidelity", "cerezo"):
            return pnp.mean(1.0 - F)
        if self.loss == "fidelity":
            return pnp.mean((1.0 - F) ** 2)
        if self.loss == "hybrid":
            return pnp.mean(self.alpha * (1.0 - F) + (1.0 - self.alpha) * (1.0 - F) ** 2)
        raise ValueError(self.loss)

    def loss_and_grad(self, p, X, y):
        pp = pnp.array(p, requires_grad=True)
        Xc, yc = np.asarray(X, float), np.asarray(y, float)   # datos: arrays NumPy simples (no se derivan)
        grad_fn = qml.grad(self._loss, argnum=0)
        g = grad_fn(pp, Xc, yc)
        return float(grad_fn.forward), np.asarray(g, float)   # .forward = valor de la pérdida en esa pasada

    def predict_proba(self, p, X, batch_size=4096):
        p = pnp.array(p, requires_grad=False)
        return np.concatenate([np.asarray(self._prob_y_z(p, np.asarray(X[i:i + batch_size], float))[0])
                               for i in range(0, len(X), batch_size)])


# Verificación contra el motor NumPy en 64 eventos de train, con parámetros genéricos (no los de arranque)
rng = np.random.default_rng(7)
Xb, yb = data["Xtr"][:64], data["ytr"][:64].astype(float)
for nombre in ("M2", "M3-lin", "M3-cua", "Z híbrida α=0.5", "Z BCE"):
    hp = cf.CONFIGS[nombre]
    circ = build_circuit("muten", len(hp.features), hp.n_layers, 1, hp.readout)
    ref = DRUModel(circ, loss=hp.loss, alpha=hp.hybrid_alpha)
    pl = ModeloPL(circ, loss=hp.loss, alpha=hp.hybrid_alpha)
    p = 0.5 * rng.normal(size=circ.n_params)
    v0, g0 = ref.loss_and_grad(p, Xb, yb)
    v1, g1 = pl.loss_and_grad(p, Xb, yb)
    dp = np.max(np.abs(ref.predict_proba(p, Xb) - pl.predict_proba(p, Xb)))
    rel = np.max(np.abs(g0 - g1)) / np.max(np.abs(g0))
    print(f"{nombre:16s} |Δp| {dp:.1e} · |Δpérdida| {abs(v0 - v1):.1e} · |Δgrad|/max|grad| {rel:.1e}")
    assert dp < 1e-10 and abs(v0 - v1) < 1e-10 and rel < 1e-5
print("PennyLane ≡ dru_core: OK")

## 5 · Entrenar con PennyLane

`entrenar_pl` es una copia del bucle de `corrida_final.train` (layerwise, dos fases, Adam nuevo por fase, early stopping
por AUC de validación que corta solo la fase en curso, checkpoint en la mejor época) con `ModeloPL` en lugar de
`DRUModel`. Reutiliza `cf.init_params`, `dru_core.Adam` y `dru_core.layer_mask`.

**Qué cambiar para la variante:**
- `CONFIG_BASE`: cualquier clave de `cf.CONFIGS`; `hp = replace(hp, ...)` para tocar épocas (`stages`), lr, capas,
  pérdida, lectura, etc.
- `N_TRAIN`, `N_VAL`: con `(5000, 1000)` se usa el split oficial. Con otros valores, `split_ampliado` **mantiene fijo el
  test oficial** (los mismos 5000 eventos) y saca train/val balanceados del resto de las `N_FILAS` filas, así los AUC de
  test siguen siendo comparables con la corrida final.
- `SEMILLAS`: la corrida final usó (101, 202, 303, 404, 505).
- `DEVICE`: `"default.qubit"` por defecto. `default.qubit` es más lento que el motor NumPy; si quieres más velocidad,
  prueba la interfaz JAX con `jax.jit` o `"lightning.qubit"` (este último no admite broadcasting de la misma forma).

`PRUEBA_RAPIDA = True` recorta épocas y eventos para comprobar que todo corre; ponlo en `False` para la corrida real.

In [ ]:
from dataclasses import replace

PRUEBA_RAPIDA = True

CONFIG_BASE = "M2"
hp = cf.CONFIGS[CONFIG_BASE]
# ejemplos:  hp = replace(hp, stages=((24, 0.02), (36, 0.01)))     # el doble de épocas
#            hp = replace(hp, loss="fidelity", readout="z")          # = M3-cua
N_TRAIN, N_VAL = 5000, 1000
SEMILLAS = (101,)
DEVICE = "default.qubit"

if PRUEBA_RAPIDA:
    hp = replace(hp, n_layers=2, stages=((2, 0.02), (1, 0.01)))
    N_TRAIN, N_VAL = 1000, 400


def split_ampliado(df, n_train, n_val, seed=2026):
    # test oficial fijo; train y val balanceados del resto de las filas de df
    if (n_train, n_val) == (5000, 1000) and len(df) >= 26_000:
        return {k: v.copy() for k, v in cf.IDX.items()}
    y = df["label"].to_numpy(int)
    libres = np.setdiff1d(np.arange(len(df)), cf.IDX["test"])
    rng = np.random.default_rng(seed)
    out = {"train": [], "val": []}
    for c in (0, 1):
        idx = rng.permutation(libres[y[libres] == c])
        nt, nv = n_train // 2, n_val // 2
        if len(idx) < nt + nv:
            raise ValueError(f"clase {c}: hay {len(idx)} eventos libres y se piden {nt + nv}; sube N_FILAS")
        out["train"].extend(idx[:nt]); out["val"].extend(idx[nt:nt + nv])
    out = {k: rng.permutation(np.asarray(v, int)) for k, v in out.items()}
    out["test"] = cf.IDX["test"].copy()
    return out


def preparar(hp, df, idx):
    # reutiliza cf.prepare (escaladores ajustados solo con train) sobre otro DataFrame / split
    viejo = cf.DF, cf.Y, cf.IDX
    try:
        cf.DF, cf.Y, cf.IDX = df, df["label"].to_numpy(int), idx
        return cf.prepare(hp)
    finally:
        cf.DF, cf.Y, cf.IDX = viejo


def entrenar_pl(hp, seed, data, device=DEVICE, verbose=True):
    # copia de corrida_final.train con ModeloPL en lugar de DRUModel
    Xtr, ytr, Xva, yva, use_val = data["Xtr"], data["ytr"], data["Xva"], data["yva"], data["use_val"]
    rng = np.random.default_rng(seed)
    circ = build_circuit("muten", len(hp.features), hp.n_layers, 1, hp.readout)
    model = ModeloPL(circ, loss=hp.loss, alpha=hp.hybrid_alpha, device=device)
    p = cf.init_params(hp, circ, seed)

    def batches():
        o = rng.permutation(len(Xtr))
        for s in range(0, len(o), hp.batch_size):
            yield Xtr[o[s:s + hp.batch_size]], ytr[o[s:s + hp.batch_size]]

    for layer in range(hp.n_layers):                      # layerwise
        opt, m = Adam(circ.n_params, lr=hp.layerwise_lr), layer_mask(circ, layer)
        for _ in range(hp.layerwise_epochs):
            for Xb, yb in batches():
                p = opt.step(p, model.loss_and_grad(p, Xb, yb)[1], m)

    best = dict(auc=-np.inf, p=p.copy(), epoch=0)
    hist, ep, stalled = [], 0, 0
    for s_i, (n_ep, lr) in enumerate(hp.stages):           # fases (adam_reset="stage")
        opt = Adam(circ.n_params); opt.lr = lr
        stalled = 0 if hp.patience_per_stage else stalled
        for _ in range(n_ep):
            ep += 1
            losses = []
            for Xb, yb in batches():
                v, g = model.loss_and_grad(p, Xb, yb); p = opt.step(p, g); losses.append(v)
            row = dict(epoch=ep, stage=s_i, lr=lr, loss=float(np.mean(losses)))
            stop_stage = False
            if use_val:
                auc = roc_auc_score(yva, model.predict_proba(p, Xva)); row["val_auc"] = auc
                if auc > best["auc"] + hp.min_delta:
                    best, stalled = dict(auc=auc, p=p.copy(), epoch=ep), 0
                else:
                    stalled += 1
                    stop_stage = stalled >= hp.patience
            hist.append(row)
            if verbose:
                print(f"  época {ep:3d} fase {s_i} pérdida {row['loss']:.4f} AUC val {row.get('val_auc', float('nan')):.4f}")
            if stop_stage:
                break
    if not use_val:
        best = dict(auc=np.nan, p=p.copy(), epoch=ep)
    return model, best, hist


assert hp.adam_reset == "stage", "entrenar_pl solo implementa adam_reset='stage' (el de todas las configuraciones)"
idx = split_ampliado(DF_GRANDE, N_TRAIN, N_VAL)
data_pl = preparar(hp, DF_GRANDE, idx)
print({k: len(v) for k, v in idx.items()}, "·", hp)

corridas = {}
for s in SEMILLAS:
    t0 = time.perf_counter()
    modelo, best, hist = entrenar_pl(hp, s, data_pl)
    corridas[s] = dict(modelo=modelo, best=best, hist=hist, segundos=time.perf_counter() - t0)
    print(f"semilla {s}: {corridas[s]['segundos']:.0f} s · mejor época {best['epoch']} · AUC val {best['auc']:.4f}")

## 6 · Evaluar y guardar

El test oficial **ya se usó una vez** para la corrida final pre-registrada. Lo que salga aquí es exploratorio: sirve
para comparar la variante con la corrida final, pero no reemplaza esos resultados ni debe usarse para elegir
hiperparámetros (para eso está la validación).

Con `PRUEBA_RAPIDA = False`, `CONFIG_BASE = "M2"`, `(N_TRAIN, N_VAL) = (5000, 1000)` y `SEMILLAS = (101,)`, el resultado
debe coincidir con el del paso 3 salvo diferencias numéricas pequeñas (mismo bucle, misma inicialización, mismo orden de lotes).
Comprobado el 15/11/2026 en una laptop (PennyLane 0.42.3): misma mejor época (27), mismo AUC de test (0.8506) y
diferencia máxima de 8e-9 en las probabilidades de test frente a `scores/69e93f3adf2f_101.npz`. Tardó 85 s, contra
11 s del motor NumPy; en Colab cuenta con algo más.

In [ ]:
from datetime import datetime

yte = data_pl["yte"]
P = np.array([c["modelo"].predict_proba(c["best"]["p"], data_pl["Xte"]) for c in corridas.values()])
for s, pt in zip(corridas, P):
    print(f"semilla {s}: AUC test {roc_auc_score(yte, pt):.4f}")
print(f"ensemble ({len(P)} semillas): AUC test {roc_auc_score(yte, P.mean(0)):.4f}")

ref = pd.read_csv(REPO / "corrida_final" / "resultados_test.csv").set_index("modelo")
if CONFIG_BASE in ref.index:
    r = ref.loc[CONFIG_BASE]
    print(f"corrida final {CONFIG_BASE}: ensemble {r.test_auc_ens:.4f} [{r.ic95_lo:.4f}, {r.ic95_hi:.4f}] · "
          f"por semilla {r.test_auc_media:.4f} ± {r.test_auc_sd:.4f}")

GUARDAR_EN_DRIVE = False
salida = pathlib.Path("/content/drive/MyDrive/susy_dru_pennylane" if (IN_COLAB and GUARDAR_EN_DRIVE)
                      else REPO / "colab" / "salidas")
if IN_COLAB and GUARDAR_EN_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
salida.mkdir(parents=True, exist_ok=True)
sello = datetime.now().strftime("%Y%m%d_%H%M%S")
for s, c in corridas.items():
    np.savez(salida / f"pl_{CONFIG_BASE.replace(' ', '_')}_{s}_{sello}.npz",
             params=c["best"]["p"], test=c["modelo"].predict_proba(c["best"]["p"], data_pl["Xte"]),
             val=c["modelo"].predict_proba(c["best"]["p"], data_pl["Xva"]),
             loss_hist=np.array([h["loss"] for h in c["hist"]]),
             val_hist=np.array([h.get("val_auc", np.nan) for h in c["hist"]]),
             hp=str(hp), n_train=N_TRAIN, n_val=N_VAL, n_filas=len(DF_GRANDE), device=DEVICE)
print("guardado en", salida)